In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 1 — O PREÇO DA CERTEZA: QUANDO PREVER DEIXA DE SER ESTIMAR

> "O mundo não é feito de átomos, é feito de histórias. Mas as decisões que o moldam, essas são feitas de categorias."
> — Antiga máxima dos Arquitetos da Verdade

Eu me lembrava vividamente do projeto anterior, a "Saga da Regressão". Meu mundo, por meses, foi um contínuo de números. O objetivo era estimar, prever um valor em uma escala infinita de possibilidades. O preço de uma casa não era um ponto fixo, mas uma dança de variáveis, um espectro de probabilidades. Eu buscava a linha de tendência, a curva que melhor se ajustava à nuvem de pontos, e minha maior preocupação era o tamanho do erro, a distância entre minha previsão e a realidade.

Hoje, o cenário é outro. A carta de Helena, com suas bordas amassadas e a caligrafia firme, repousa ao lado do meu teclado. Ela não é um ponto em um gráfico de dispersão. Ela é uma pessoa. E o problema que a afetou não era sobre "**quão perto**", mas sobre "**qual**". Benigno ou Maligno. Sim ou Não. Uma decisão binária, com consequências absolutas. O erro aqui não é uma questão de grau, mas de identidade: é a diferença entre uma vida de alívio e uma jornada de tratamento.

Diante de mim, a tela exibe as primeiras linhas do projeto OncoClassify 2.0. Me lembrei dos oncologistas que usam a ferramenta. Eles não querem uma probabilidade de 72,3%. Eles precisam de uma decisão, uma categoria. E essa necessidade de certeza, essa passagem do contínuo para o discreto, muda tudo. A filosofia por trás do código, **a arquitetura da verdade** que eu preciso construir, deve ser fundamentalmente diferente.

## A Mudança de Paradigma: De Regressão Para Classificação

No Volume 1 desta saga, o objetivo era prever um valor numérico contínuo: o preço de um imóvel. O sucesso era medido pela proximidade das previsões aos valores reais. Se prevíamos R\$ 305.000 para uma casa vendida por R\$ 300.000, o erro era de R\$ 5.000. Um erro menor significava um modelo melhor.

Agora entramos no domínio da **classificação**. O objetivo não é mais prever um número em uma escala, mas atribuir uma etiqueta, uma categoria, a uma observação. A pergunta central muda de "**Quanto?**" para "**Qual?**".

Formalmente, um problema de classificação consiste em aprender uma função de mapeamento f(X) → y, onde X é o conjunto de variáveis de entrada (*features*) e y é uma variável de saída categórica de um conjunto finito de classes. No **Wisconsin Breast Cancer Dataset**, as classes são {Maligno, Benigno}. Como há apenas duas, este é um problema de **classificação binária**.

| Característica | Regressão (Vol. 1) | Classificação (Vol. 2) |
|---|---|---|
| Objetivo | Prever o preço de uma casa | Diagnosticar um tumor |
| Variável Alvo | Contínua (ex.: R\$ 250.000) | Categórica (Maligno, Benigno) |
| Pergunta Chave | "Quanto custa?" | "Qual é o diagnóstico?" |
| Métrica de Erro | Distância (ex.: Erro Quadrático Médio) | Contagem de erros (ex.: Matriz de Confusão) |
| Visualização | Linha de tendência | Fronteira de decisão separando classes |

Enquanto na regressão buscávamos uma linha que passasse *através* dos pontos, na classificação buscamos uma superfície que *separe* os pontos em grupos distintos. Essa superfície é a **fronteira de decisão**.

> **📘 PARA SABER — Binária, Multiclasse e Multirrótulo**
>
> **Binária:** decisão entre duas classes (Sim/Não, Maligno/Benigno). É o tipo mais fundamental e o foco absoluto deste livro.
>
> **Multiclasse:** três ou mais classes mutuamente exclusivas (ex.: subtipos de tumor). Retomamos o tema no Capítulo 32.
>
> **Multirrótulo:** cada observação pode pertencer a várias classes ao mesmo tempo (ex.: um artigo marcado como "Política" *e* "Economia").

## Primeiro Contato Com os Dados

Todo o projeto se apoia num único ponto de verdade: o módulo `oncoclassify_utils`. Ele centraliza constantes, o carregamento dos dados e a curadoria, seguindo o princípio **DRY** (*Don't Repeat Yourself*): cada decisão vive num só lugar. Começamos importando-o e conferindo a configuração do projeto.

#### Código 1.1: Configuração do Projeto


In [ ]:
# Ponto único de verdade do projeto: constantes, dados e utilitários.
from oncoclassify_utils import print_project_info

print_project_info()

PROJETO ONCOCLASSIFY 2.0
   A Arquitetura da Verdade - A Saga da Classificacao
   RANDOM_STATE:      42
   TEST_SIZE:         0.2
   CUSTO_FN:          100 (Falso Negativo)
   CUSTO_FP:          10 (Falso Positivo)
   Base canonica:     600 amostras
   Features (modelo): 30 morfologicas
   Convencao alvo:    0 = Maligno | 1 = Benigno


Duas convenções ficam fixas desde já e valem para o livro inteiro: **0 = Maligno, 1 = Benigno**, sendo a classe Maligna a nossa **classe positiva de interesse**; e os custos assimétricos `CUSTO_FN = 100` e `CUSTO_FP = 10`, que formalizam a intuição de que deixar passar um câncer é muito pior do que um alarme falso — usaremos esses números a partir do Capítulo 16.

Antes de qualquer modelo, precisamos conhecer os dados como eles realmente chegam: **sujos**. Foi exatamente o descuido com a qualidade dos dados que condenou o OncoClassify 1.0.

#### Código 1.2: O Dataset Como Ele Vem (Bruto)


In [ ]:
from oncoclassify_utils import df_raw

# df_raw e o dataset MODIFICADO cru -- de proposito cheio de problemas reais.
print(f"Dimensoes: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas\n")

# Valores ausentes: onde faltam dados?
print("Valores ausentes (colunas com pelo menos 1 nulo):")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Duplicatas e formatos inconsistentes -- os vilões silenciosos da qualidade.
print(f"\nLinhas duplicadas: {df_raw.duplicated().sum()}")
print("Formatos em 'patient gender':", list(df_raw['patient gender'].unique()))
print("Formatos em 'family history':", list(df_raw['family history'].dropna().unique()))

Dimensoes: 610 linhas x 41 colunas

Valores ausentes (colunas com pelo menos 1 nulo):
patient age             92
biopsy time             54
family history         119
genetic test result    600
dtype: int64

Linhas duplicadas: 10
Formatos em 'patient gender': ['F', 'female', 'Female', 'FEMALE', 'f', 'Fem', 'Woman']
Formatos em 'family history': ['0', 'NO', '1', 'true', 'True', 'no', 'yes', 'YES']


> **⚠️ ATENÇÃO — Os defeitos que este dataset esconde**
>
> Em poucas linhas, o dataset bruto já revela quatro patologias clássicas de dados do mundo real:
>
> **1. Duplicatas** — 10 linhas repetidas, que inflariam artificialmente algumas classes.
>
> **2. Valores ausentes** — de leves (`biopsy time`, 54) a fatais: `genetic test result` está **600 de 610** vazia, ou seja, é praticamente inútil.
>
> **3. Categóricas inconsistentes** — `patient gender` aparece como `F`, `f`, `FEMALE`, `Woman`… (e, no fim, é sempre "feminino"); `family history` mistura `0/1`, `yes/no`, `true/True`.
>
> **4. Colunas que não deveriam existir num modelo** — identificadores (`patient code`, `scan_id`, `session code`) e duas colunas de aparência inocente, `diagnosis progression score` e `post evaluation flag`, que guardam uma armadilha perigosa. **Voltaremos a elas no Capítulo 4.**

A curadoria — a decisão fundamentada sobre o que manter e o que descartar — está encapsulada na função `limpar_dataset`, transparente e reexecutável. Ela remove duplicatas, descarta identificadores, colunas de vazamento e colunas clínicas fracas ou vazias, e mantém apenas as **30 features morfológicas** (as medidas do núcleo celular) mais o alvo. O resultado é a nossa **base canônica**, `df_full` — a mesma em todos os capítulos.

#### Código 1.3: Da Base Bruta à Base Canônica


In [ ]:
from oncoclassify_utils import df_raw, limpar_dataset

# retornar_relatorio=True devolve tambem um resumo do que a curadoria fez.
base, relatorio = limpar_dataset(df_raw, retornar_relatorio=True)

for chave, valor in relatorio.items():
    print(f"  {chave}: {valor}")

  linhas_brutas: 610
  duplicatas_removidas: 10
  linhas_canonicas: 600
  colunas_descartadas: ['patient code', 'scan_id', 'session code', 'diagnosis progression score', 'post evaluation flag', 'patient age', 'patient gender', 'family history', 'biopsy time', 'genetic test result']
  features_modelo: 30


De 610 linhas sujas para **600 amostras limpas** e 30 features de sinal genuíno. Note que descartamos `diagnosis progression score` e `post evaluation flag` — as duas colunas "suspeitas". Elas *parecem* preditores excelentes, e é justamente por isso que são perigosas: no Capítulo 4 vamos provar, com número na tela, por que incluí-las seria repetir o erro que custou a Helena.

## A Variável Alvo: Comece Sempre Por Aqui

A primeira análise de qualquer projeto de classificação é sobre a variável que se quer prever. Antes de olhar qualquer *feature*, olhamos o alvo: quantos casos de cada classe existem?

#### Código 1.4: Distribuição das Classes


In [ ]:
import pandas as pd
from oncoclassify_utils import df_full, color_map, apply_hatches_bars
import seaborn as sns, matplotlib.pyplot as plt

resumo = pd.DataFrame({
    "quantidade": df_full["diagnosis_label"].value_counts(),
    "proporcao":  df_full["diagnosis_label"].value_counts(normalize=True).round(3),
})
print(resumo)

ax = sns.countplot(data=df_full, x="diagnosis_label", hue="diagnosis_label",
                   palette=color_map, order=["Benigno", "Maligno"], legend=False)
apply_hatches_bars(ax, class_order=["Benigno", "Maligno"])
ax.set_title("Distribuição das Classes (Benigno vs Maligno)")
ax.set_xlabel("Diagnóstico"); ax.set_ylabel("Quantidade")
plt.tight_layout(); plt.show()

                 quantidade  proporcao
diagnosis_label
Benigno                 380      0.633
Maligno                 220      0.367


![Distribuição das classes](media/figura_01_1.png)

Temos **380 casos benignos (63,3%)** e **220 malignos (36,7%)**. O dataset é **desbalanceado** — não severamente, mas o suficiente para exigir cuidado. Um modelo preguiçoso que simplesmente "chutasse benigno" para todos acertaria 63,3% das vezes: um número que *parece* razoável e é clinicamente catastrófico, porque erraria **todos** os casos de câncer. Essa miopia estatística é o coração do próximo capítulo.

#### Código 1.5: Como as Classes se Separam nas Features


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from oncoclassify_utils import df_full, color_map

features = ["mean radius", "mean texture", "mean area"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, feature in zip(axes, features):
    # Histograma sobreposto por classe: onde os dois grupos se distinguem?
    sns.histplot(data=df_full, x=feature, hue="diagnosis_label",
                 palette=color_map, alpha=0.6, ax=ax, legend=(ax is axes[0]))
    ax.set_title(f"Distribuição de {feature}")

plt.suptitle("Distribuição de Variáveis por Classe (Benigno × Maligno)", fontsize=15)
plt.tight_layout(); plt.show()

![Histogramas por classe](media/figura_01_2.png)

Tumores **malignos** concentram-se em valores maiores de `mean radius`, `mean texture` e `mean area`; os **benignos**, em valores menores. Essas variáveis, sozinhas, já contam parte da história. Mas repare na **zona de interseção**, onde as duas distribuições se sobrepõem: é ali que moram os casos ambíguos — é ali que estava a Helena. Nenhuma *feature* isolada resolve essa zona; construir um modelo que a atravesse com segurança é o trabalho de todo o resto do livro.

> **💡 DICA — Comece sempre pela variável alvo**
>
> Em qualquer projeto, a primeira análise é sobre o que você quer prever: sua distribuição, seu desbalanceamento, seu significado no mundo real. Entender o alvo guia todas as decisões seguintes. É a fundação de um modelo bem-sucedido.

> **📌 CHECKLIST DO CAPÍTULO 1**
>
> ✅ Entendeu a diferença entre **Regressão** ("quanto") e **Classificação** ("qual").
>
> ✅ Reconheceu os quatro defeitos de qualidade do dataset bruto: duplicatas, nulos, categóricas inconsistentes e colunas indevidas.
>
> ✅ Executou a curadoria e chegou à base canônica: **600 amostras, 30 features morfológicas**.
>
> ✅ Identificou o desbalanceamento: **380 benignos (63,3%)** e **220 malignos (36,7%)**.
>
> ✅ Refletiu sobre por que a acurácia pode enganar — um modelo pode acertar 63% e ser um fracasso clínico.
>
> **⚠️ CRÍTICO:** Em classificação, o contexto define o que é um "bom" modelo. Errar um diagnóstico maligno é muito pior do que errar um benigno. Esse custo assimétrico é o tema do próximo capítulo.

O primeiro passo foi dado. Os dados agora vivem dentro do meu computador não como uma lista fria de números, mas como uma responsabilidade. O gráfico do desbalanceamento não é apenas uma estatística: é um lembrete de que a saúde é mais comum que a doença — mas é a doença que exige nossa máxima vigilância.

Eu havia recapitulado a fronteira entre os dois mundos do *machine learning*. Agora precisava dissecar a própria natureza do erro, para entender que, neste novo paradigma, nem todos os erros são criados iguais.
